In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.default.bronze_telemetry")
robots = spark.table("workspace.default.robots")

# 1. Remove duplicates: one row per robot per timestamp
# A duplicate = same robot at the same time, so those two columns define a unique row
deduped = bronze.dropDuplicates(["robot_id", "ts"])

# 2. Bring in each robot's model; Join on the column both tables share
with_model = deduped.join(robots.select("robot_id", "model"), "robot_id", "left")

# 3. Convert RX-300 temperatures from Fahrenheit to Celsius
silver = with_model.withColumn(
    "motor_temp_c",
    F.when(F.col("model") == "RX-300", (F.col("motor_temp_c") - 32) * 5 / 9)
     .otherwise(F.col("motor_temp_c"))
)

silver.write.mode("overwrite").saveAsTable("workspace.default.silver_telemetry")

In [0]:
print("bronze:", bronze.count())
print("silver:", silver.count())
display(silver.groupBy("model").agg(F.avg("motor_temp_c").alias("avg_temp")))


bronze: 4508231
silver: 4464000


model,avg_temp
RX-300,58.00357228278925
RX-100,58.01542569764917
RX-200,58.00068218094408
